### Resultados do aprendizado

Após este laboratório, os estudantes serão capazes de:
- implementar um pipeline RAG 
- analizar saídas LLM criticamente
- avaliar a confiabilidade dos sistemas

In [ ]:
%pip install -q google-genai sentence-transformers faiss-cpu

In [ ]:
import google.genai as genai
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [ ]:
API_KEY = "GEMINI_API_KEY"

client = genai.Client(api_key=API_KEY)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explique overfitting em português"
)

print(response.text)

In [ ]:
documents = [
    "A otimização da cadeia de suprimentos envolve a redução de custos, mantendo os níveis de serviço.",
    "A gestão de estoque é fundamental para equilibrar a oferta e a demanda.",
    "Os custos de transporte são uma componente importante das despesas logísticas.",
    "A previsão da demanda ajuda a reduzir a incerteza nas cadeias de abastecimento.",
    "As relações com os fornecedores influenciam a eficiência e a resiliência."
]

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embedder.encode(documents)

In [ ]:
index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(np.array(doc_embeddings))

In [ ]:
def retrieve(query, k=2):
    q_emb = embedder.encode([query])
    distances, indices = index.search(np.array(q_emb), k)
    return [documents[i] for i in indices[0]]

In [ ]:
def ask_llm(prompt):
    return client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

In [ ]:
query = "Como reduzir os custos da cadeia de suprimentos?"

simple = ask_llm(query)

structured = ask_llm(f"Explique claramente: {query}")

context = retrieve(query)
rag_prompt = f"""
Use o seguinte contexto:
{context}

Resposta:
{query}
"""
rag = ask_llm(rag_prompt)

print("SIMPLES:\n", simple)
print("\nESTRUTURADA:\n", structured)
print("\nRAG:\n", rag)

In [ ]:
print("""
Compare:

- Qual é mais factual?
- Qual usa melhor contexto?
- Qual é mais confiável?
""")